# 04. Simulation validation: does the method work when the truth is known?

**The question.** Before trusting any estimate on market data, does each estimator recover a known exposure, does the variation test have the right false-positive rate, and can the full hedging procedure detect a real refinancing effect while staying quiet when there is none?

**Why a portfolio manager cares.** A backtest on 12 years of data contains only a few refinancing cycles. The manager needs to know how large an improvement the procedure could reliably detect, and how often it would report a gain that is not there.

**Learning objectives**

1. Measure the bias and interval coverage of the monthly realized-duration estimator.
2. Check how often Cochran's Q rejects a constant duration when the duration really is constant.
3. Refit the entire pipeline in many synthetic worlds with and without state dependence.
4. Report the false-positive rate of the cost verdict and its power at a realistic effect size.

| | |
|---|---|
| Inputs | `mbshedge.synthetic` worlds (no market data) |
| Outputs | `results/simulation/*.csv`, `results/simulation/power_summary.json`, figures `04_*` |
| Run after | nothing (independent of the real data) |
| Approximate runtime | about 6 minutes (about 1 minute in smoke mode) |

> **Every number in this notebook comes from synthetic data with a known truth.** These worlds test the *procedure*. They say nothing about whether the effect exists in MBS. Their yield path is loosely shaped like 2006–2026 history. The duration either responds to refinancing with a lag of a few weeks (scenario "state"), stays constant at 4.5 years (scenario "constant"), or moves about as much as in the state scenario but along its own random path that has nothing to do with refinancing (scenario "unrelated").

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [ ]:
from mbshedge import simulation as sim
from mbshedge import synthetic as syn
sdir = paths.results / "simulation"; sdir.mkdir(parents=True, exist_ok=True)
n_worlds = cfg["smoke"]["simulation_worlds"] if MODE == "smoke" else cfg["simulation"]["worlds"]
n_recovery = 2 if MODE == "smoke" else cfg["simulation"]["recovery_worlds"]
n_qnull = 6 if MODE == "smoke" else cfg["simulation"]["q_null_worlds"]
q_reps = 60 if MODE == "smoke" else cfg["simulation"]["q_null_bootstrap_reps"]
reps = min(cfg["simulation"]["bootstrap_reps"], EXP["evaluation"]["bootstrap_reps"])
base_seed = EXP["seed"] + 1000
print(f"worlds per scenario: {n_worlds}, recovery worlds: {n_recovery}, Q-null worlds: {n_qnull}, bootstrap draws per world: {reps}")

## Study 1. Does the monthly estimator recover the true duration?

Within a month, the regression $r_t = a + b_0\,\Delta y_t + b_1\,\Delta y_{t-1} + e_t$ estimates an average of the daily durations, weighted by how much each day's yield moved (days with big moves carry more information). The target is therefore $\bar D_m = \sum_t \Delta y_t^2 D_{t-1} / \sum_t \Delta y_t^2$. We compare the estimate with that target and check whether the nominal 95% interval $\hat D_m \pm 1.96\,\text{se}$ covers it about 95% of the time.

In [ ]:
rec = sim.recovery_study(cfg, EXP, [base_seed + i for i in range(n_recovery)])
rec.to_csv(sdir / "recovery_months.csv", index=False)
summary1 = {"months": int(len(rec)), "mean_error_years": float(rec["error"].mean()),
            "rmse_years": float(np.sqrt((rec["error"] ** 2).mean())), "median_se_years": float(rec["se"].median()),
            "coverage_95": float(rec["covered_95"].mean())}
print(json.dumps(summary1, indent=1))

In [ ]:
one = rec[rec["seed"] == rec["seed"].iloc[0]].copy()
one["date"] = pd.PeriodIndex(one["month"], freq="M").to_timestamp(how="end")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [2.2, 1]})
axes[0].errorbar(one["date"], one["D"], yerr=1.96 * one["se"], fmt="o", ms=3, color="#2a78d6", ecolor="#9ec5f4", lw=0.8, label="estimate with 95% interval")
axes[0].plot(one["date"], one["true_weighted"], color=pt.INK, lw=1.4, label="true information-weighted duration")
axes[0].set_ylabel("years"); axes[0].set_title("One synthetic world: monthly estimates against the truth"); axes[0].legend(loc="upper left")
axes[1].hist(rec["error"] / rec["se"], bins=40, color="#2a78d6", alpha=0.8)
axes[1].set_title("Standardized errors (should look standard normal)"); axes[1].set_xlabel("(estimate - truth) / se")
pt.save_figure(fig, "04_recovery", paths, data=rec, mode="smoke")   # simulation figures are always synthetic
plt.show()

## Study 2. Does the variation test reject too often when duration is constant?

Cochran's Q compares the spread of the monthly estimates with their own standard errors. Those standard errors come from about 21 daily observations with fat-tailed noise, so the textbook $\chi^2$ reference can be badly off: if the error bars are too narrow, Q looks large even when nothing varies.

The project therefore uses a **constant-duration residual bootstrap**: fit one duration to the whole window, rebuild each month's returns from that fit plus residuals resampled within the same month (in 5-day blocks), re-estimate every month, and see how often the bootstrap Q exceeds the observed one. Here we check both p-values in worlds where the true duration never changes, so a well-calibrated test should reject about 5% of the time.

In [ ]:
qn = sim.q_null_study(cfg, EXP, [base_seed + 500 + i for i in range(n_qnull)],
                      (EXP["sample"]["first_month"], EXP["sample"]["development_end"]), reps=q_reps)
qn.to_csv(sdir / "q_null.csv", index=False)
q_reject_naive = float((qn["p_chi2_naive"] < 0.05).mean())
q_reject = float((qn["p_bootstrap"] < 0.05).mean())
print(f"Constant-duration worlds: {len(qn)}")
print(f"  rejection rate at 5%, naive chi-square p-value:   {q_reject_naive:.2f}")
print(f"  rejection rate at 5%, constant-duration bootstrap: {q_reject:.2f}   (target 0.05)")
print(f"  median robust tau under the null: {qn['tau_robust'].median():.3f} years (the true value is 0)")

## Study 3. The full pipeline in many worlds

Each world is analysed from scratch exactly as the real data will be: the EWMA half-life is chosen on development data, every model is fit walk-forward, the common backtest runs at the base settings, and the paired bootstrap and the declared cost verdict are applied to the test period.

* In **state** worlds duration follows the share of borrowers in the money with a lag of a few weeks, the situation in which refinancing information *should* help.
* In **constant** worlds the true duration never changes. No updating hedge should beat the static one, so these worlds are the reference null for the comparisons against the static hedge.
* In **unrelated** worlds duration moves as much, and as persistently, as in the state worlds, but independently of refinancing. Updating helps there, and refinancing information should add nothing beyond updating, so these worlds are the reference null for the hybrid against the updating hedge.

For each comparison, the share of its reference null worlds that pass the survives-costs verdict is the **false-positive rate**, and the share of state worlds that pass it is the **power** at the effect size those worlds produce. The power is measured directly rather than computed from the spread of the null worlds: when the hybrid really differs from the updating hedge, the test is much noisier than when the two nearly coincide, so a formula based on the null spread would promise more power than the test has. The economic model against the updating hedge has no reference null, because the two are not nested: with a constant duration the economic model collapses to a near-constant forecast that genuinely beats the noisier updating estimate, and with an unrelated duration it cannot track the moves at all. The table reports how often it wins in each kind of world instead.

In [ ]:
import time
t0 = time.time()
worlds = {}
for name, offset in [("state", 2000), ("constant", 4000), ("unrelated", 6000)]:
    worlds[name] = sim.pipeline_worlds(cfg, EXP, [base_seed + offset + i for i in range(n_worlds)], name, reps)
    worlds[name].to_csv(sdir / f"worlds_{name}.csv", index=False)
print(f"{3 * n_worlds} worlds in {time.time() - t0:.0f} s")

In [ ]:
rows = []
for key in ["H_vs_U", "E_vs_U", "U_vs_S", "H_vs_S"]:
    rows.append(sim.power_summary(worlds, key))
power = pd.DataFrame(rows).set_index("comparison")
(sdir / "power_summary.json").write_text(json.dumps({"recovery": summary1, "q_null_rejection_rate_5pct": q_reject,
                                                    "q_null_rejection_rate_naive_chi2": q_reject_naive,
                                                    "comparisons": power.reset_index().to_dict(orient="records")}, indent=2))
show = ["reference_null", "null_share_survives_costs", "state_mean_variance_reduction", "state_median_ci_halfwidth",
        "state_share_survives_costs", "constant_mean_variance_reduction", "constant_share_survives_costs",
        "unrelated_mean_variance_reduction", "unrelated_share_survives_costs"]
power[show].T.round(3)

In [ ]:
from matplotlib.ticker import MaxNLocator
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
null_label = {"unrelated": "unrelated worlds (duration moves, not with refinancing)", "constant": "constant worlds (duration never moves)"}
for ax, key, title in [(axes[0], "H_vs_U", "Hybrid vs updating"), (axes[1], "U_vs_S", "Updating vs static")]:
    ref = power.loc[key, "reference_null"]
    null, state = worlds[ref], worlds["state"]
    bins = np.linspace(min(null[f"vr_{key}"].min(), state[f"vr_{key}"].min()), max(null[f"vr_{key}"].max(), state[f"vr_{key}"].max()), 30)
    ax.hist(null[f"vr_{key}"], bins=bins, color="#8c8c8c", alpha=0.75, label=null_label[ref])
    ax.hist(state[f"vr_{key}"], bins=bins, color="#1baf7a", alpha=0.6, label="state worlds (duration follows refinancing)")
    ax.axvline(0, color=pt.INK, lw=0.8)
    target = EXP["evaluation"]["practical_variance_reduction_vs_U" if key.endswith("_U") else "practical_variance_reduction_vs_S"]
    ax.axvline(target, color=pt.INK, ls=":", lw=1.2, label=f"practical target ({target:.0%})")
    ax.set_title(f"{title}: test-period variance reduction"); ax.set_xlabel("variance reduction"); ax.set_ylabel("worlds")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True)); ax.legend(fontsize=8)
pt.save_figure(fig, "04_power", paths, data=worlds, mode="smoke")   # simulation figures are always synthetic
plt.show()

## What the simulations say about the procedure

In [ ]:
hu = power.loc["H_vs_U"]
print(f"1. Estimator: mean error {summary1['mean_error_years']:+.3f} years, RMSE {summary1['rmse_years']:.2f} years, "
      f"95% interval coverage {summary1['coverage_95']:.0%} (target 95%).")
print(f"2. Under a constant duration the naive chi-square version of Cochran's Q rejects {q_reject_naive:.0%} of the time at 5%, "
      f"while the constant-duration bootstrap rejects {q_reject:.0%} ({int(round(q_reject * len(qn)))} of {len(qn)} worlds, target 5%). "
      f"With {len(qn)} worlds each world moves the rate by {1 / len(qn):.0%}. Notebooks 05 and 08 use the bootstrap p-value.")
print(f"3. Hybrid vs updating: in unrelated worlds (duration moves, but not with refinancing) the hybrid passed the declared "
      f"survives-costs verdict in {hu['null_share_survives_costs']:.0%} of worlds (the false-positive rate). In state worlds the mean "
      f"variance reduction was {hu['state_mean_variance_reduction']:.1%}, the typical 95% interval spanned about "
      f"plus or minus {hu['state_median_ci_halfwidth']:.1%}, and the verdict passed in {hu['state_share_survives_costs']:.0%} of worlds "
      f"(the power for an effect of that size).")
us = power.loc["U_vs_S"]
print(f"4. Updating vs static: in constant worlds the updating hedge passed the verdict in {us['null_share_survives_costs']:.0%} of worlds. "
      f"In unrelated worlds, where duration moves without refinancing, it passed in {us['unrelated_share_survives_costs']:.0%}.")
if hu["state_share_survives_costs"] < 0.8:
    print(f"So a single test period misses a real gain of about {hu['state_mean_variance_reduction']:.0%} in roughly "
          f"{1 - hu['state_share_survives_costs']:.0%} of worlds. ", end="")
print("Notebook 08 calls a no-gain result inconclusive when its 95% interval still reaches the 5% target, "
      "so a missed gain is not reported as evidence of no value.")

## What this does not show

* Synthetic worlds share the model structure the procedure assumes (a lagged response of duration to the share in the money). Real MBS duration also moves with spreads, index composition, and Federal Reserve purchases.
* The bootstrap intervals in notebook 08 hold the positions fixed. These simulations add the variability from refitting every model, so they are the better guide to how noisy the whole procedure is.